# Stellar contamination grid: epsilon(lambda) for two-spot models

This notebook computes the wavelength-dependent stellar contamination factor
`epsilon(lambda)` for the sub-Neptune spectra. The output is the set of
`TLS/epsilon_*.txt` curves used by the stellar-contamination workflow.

The grid varies:
- Photosphere temperature: `T_S_VALUES`
- Spot covering fraction: `F_SPOT_VALUES`
- Facula covering fraction: `F_FAC_VALUES`

For each case, the notebook writes a two-column file with wavelength in microns
and the corresponding contamination factor.

In [ ]:
from __future__ import annotations

import itertools
from collections import defaultdict
from pathlib import Path
from typing import Iterable, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from POSEIDON.constants import R_Sun
from POSEIDON.core import create_star
from POSEIDON.stellar import stellar_contamination

In [ ]:
# Stellar parameters for the K2-18b analogue host star.
R_S: float = 0.468 * R_Sun
LOG_G_S: float = 4.6
MET_S: float = 0.0

# Photosphere temperatures (K).
T_S_VALUES: np.ndarray = np.array([3400.0, 3500.0, 3600.0], dtype=float)

# Wavelength grid (microns). The values in waves.txt are used without resampling.
WL_UM: np.ndarray = np.loadtxt("waves.txt", dtype=float)

# Covering-fraction grids.
F_SPOT_VALUES: np.ndarray = np.linspace(0.0, 0.3, 4)  # 0.00, 0.10, 0.20, 0.30
F_FAC_VALUES: np.ndarray = np.linspace(0.0, 0.4, 4)   # 0.00, 0.133..., 0.266..., 0.40

# Use all available workers by default, following the common n_jobs=-1 convention.
N_JOBS: int = -1

OUT_DIR = Path("TLS")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Worker definition

Each worker receives one `(T_s, f_spot, f_fac)` tuple and returns the same grid
coordinates plus `epsilon(lambda)`.

The spot and facula temperatures follow the Rackham et al. (2017) prescription:
- `T_spot = 0.86 * T_s`
- `T_fac = T_s + 100 K`

In [ ]:
def compute_epsilon_for_case(
    args: Tuple[float, float, float],
) -> Tuple[float, float, float, np.ndarray]:
    """Compute epsilon(lambda) for one stellar-contamination grid point."""
    t_s, f_spot, f_fac = args

    t_spot = 0.86 * t_s
    t_fac = t_s + 100.0

    star = create_star(
        R_S,
        t_s,
        LOG_G_S,
        MET_S,
        stellar_grid="phoenix",
        stellar_contam="two_spots",
        f_spot=f_spot,
        T_spot=t_spot,
        f_fac=f_fac,
        T_fac=t_fac,
    )

    epsilon = stellar_contamination(star, WL_UM)
    return t_s, f_spot, f_fac, np.asarray(epsilon, dtype=float)

## Task grid and parallel execution

The task list contains every combination of `T_s`, `f_spot`, and `f_fac`.
`joblib.Parallel` evaluates the independent `epsilon(lambda)` cases, then the
curves are saved and summarized after all workers finish.

In [ ]:
def build_tasks(
    t_s_values: Sequence[float],
    f_spot_values: Sequence[float],
    f_fac_values: Sequence[float],
) -> List[Tuple[float, float, float]]:
    """Create every `(T_s, f_spot, f_fac)` grid point."""
    tasks: List[Tuple[float, float, float]] = []
    for t_s in t_s_values:
        for f_spot, f_fac in itertools.product(f_spot_values, f_fac_values):
            tasks.append((float(t_s), float(f_spot), float(f_fac)))
    return tasks


def run_parallel(
    tasks: Iterable[Tuple[float, float, float]],
    n_jobs: int = N_JOBS,
) -> List[Tuple[float, float, float, np.ndarray]]:
    """Evaluate the stellar-contamination grid with joblib."""
    return Parallel(n_jobs=n_jobs)(
        delayed(compute_epsilon_for_case)(task)
        for task in tasks
    )


def regroup_by_temperature(
    results: Sequence[Tuple[float, float, float, np.ndarray]]
) -> dict[float, list[tuple[float, float, np.ndarray]]]:
    """Group evaluated curves by photosphere temperature."""
    by_t_s: dict[float, list[tuple[float, float, np.ndarray]]] = defaultdict(list)
    for t_s, f_spot, f_fac, epsilon in results:
        by_t_s[t_s].append((f_spot, f_fac, epsilon))
    return by_t_s

In [ ]:
def save_epsilon_curve(
    out_dir: Path,
    t_s: float,
    f_spot: float,
    f_fac: float,
    wl_um: np.ndarray,
    epsilon: np.ndarray,
) -> Path:
    """Save one `(wavelength, epsilon)` curve with a deterministic filename."""
    filename = out_dir / f"epsilon_T{int(t_s)}_fspot{f_spot:.3f}_ffac{f_fac:.3f}.txt"
    np.savetxt(filename, np.column_stack((wl_um, epsilon)))
    return filename


def plot_epsilon_summary(
    wl_um: np.ndarray,
    by_t_s: dict[float, list[tuple[float, float, np.ndarray]]],
) -> None:
    """Plot individual curves and the mean curve for each photosphere temperature."""
    color_map = {
        3400.0: "tab:blue",
        3500.0: "tab:orange",
        3600.0: "tab:green",
    }

    plt.figure(figsize=(12, 8))

    for t_s in sorted(by_t_s.keys()):
        cases = by_t_s[t_s]
        color = color_map.get(t_s, "tab:gray")

        all_eps = []
        for f_spot, f_fac, epsilon in cases:
            all_eps.append(epsilon)
            plt.plot(wl_um, epsilon, color=color, alpha=0.25, linewidth=0.8)

        all_eps_arr = np.asarray(all_eps, dtype=float)
        epsilon_mean = all_eps_arr.mean(axis=0)

        plt.plot(
            wl_um,
            epsilon_mean,
            color=color,
            alpha=0.9,
            linewidth=2.0,
            label=f"T_s = {int(t_s)} K (mean)",
        )

    plt.xlabel("Wavelength [um]")
    plt.ylabel("Stellar contamination epsilon(lambda)")
    plt.title("Stellar contamination for multiple T_s and covering fractions")
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == "__main__":
    tasks = build_tasks(T_S_VALUES, F_SPOT_VALUES, F_FAC_VALUES)

    results = run_parallel(tasks)
    by_t_s = regroup_by_temperature(results)

    for t_s, cases in by_t_s.items():
        for f_spot, f_fac, epsilon in cases:
            save_epsilon_curve(
                out_dir=OUT_DIR,
                t_s=t_s,
                f_spot=f_spot,
                f_fac=f_fac,
                wl_um=WL_UM,
                epsilon=epsilon,
            )

    plot_epsilon_summary(WL_UM, by_t_s)